In [ ]:
# Install libraries for PDF, Word, and OCR
!pip install pymupdf python-docx pytesseract pdf2image
# Install Tesseract OCR engine and Poppler (for PDF to image conversion)
!apt-get install -y tesseract-ocr poppler-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 6.0 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 1 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 186 kB in 1s (225 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 117528 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing

In [ ]:
import fitz  # PyMuPDF
import docx
import pytesseract
from PIL import Image
from google.colab import files
import io

def extract_from_pdf(file_path):
    """Extracts text from a standard, searchable PDF."""
    text = ""
    with fitz.open(file_path) as doc:
        for page in doc:
            text += page.get_text()
    return text

def extract_from_docx(file_path):
    """Extracts text from a .docx file."""
    doc = docx.Document(file_path)
    return "\n".join([para.text for para in doc.paragraphs])

def extract_from_image(file_path):
    """Extracts text from an image (PNG, JPG) using OCR."""
    return pytesseract.image_to_string(Image.open(file_path))

# --- File Upload Trigger ---
uploaded = files.upload()

for filename in uploaded.keys():
    print(f"\n--- Processing: {filename} ---")

    if filename.endswith('.pdf'):
        # Try standard extraction first
        content = extract_from_pdf(filename)
        # If the PDF is scanned (empty text), you'd need OCR (pdf2image)
        if not content.strip():
            print("Note: This PDF appears to be an image. Use OCR methods.")

    elif filename.endswith('.docx'):
        content = extract_from_docx(filename)

    elif filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        content = extract_from_image(filename)

    else:
        content = "Unsupported file format."

    print(content[:500] + "...") # Print first 500 characters

Saving CE_Syllabus_BTech_AY_2021_updated2.pdf to CE_Syllabus_BTech_AY_2021_updated2.pdf

--- Processing: CE_Syllabus_BTech_AY_2021_updated2.pdf ---
 
 
BACHELOR OF TECHNOLOGY 
COMPUTER ENGINEERING 
 
 
 
 
 
 
 
 
 
SYLLABI BOOK 
(2021 – 2022) 
 
 
 
 
 
 
 
 
 
 
 
Department of Computer Engineering 
Faculty of Technology 
Dharmsinh Desai University 
Nadiad – 387 001, Gujarat, India. 
 
http://www.ddu.ac.in 
 
 
 
 
B. Tech. Semester I 
 
SUBJECTS 
Teaching Scheme 
Total
Credit
Examination Scheme (Marks)
Th 
Tut 
Pract 
Th 
Sess
TW
Prac
Total
Mathematics I 
3 
1 
0 
4 
4.0 
60 
40 
0 
- 
100 
Basic Electrical Engineering for ICT 
3 
1 
2 
...


In [ ]:
#

In [ ]:
# Upload files to colab, from colab, upload files to supabase


# Task
Install the Supabase client library, configure it with the necessary credentials, and then upload the files that were previously extracted in Colab to a Supabase Storage bucket.

## Install Supabase client

### Subtask:
Install the necessary `supabase-py` library in the Colab environment.


**Reasoning**:
To install the `supabase-py` library, I need to execute a pip install command in a code cell.



In [ ]:
pip install supabase-py

**Reasoning**:
The previous attempt to install `supabase-py` failed because the package name was incorrect. The correct package name for the Supabase Python client library is `supabase`.



In [ ]:
pip install supabase

## Configure Supabase credentials

### Subtask:
Set up environment variables or direct inputs for your Supabase URL and anonymous key for secure access.


To get your Supabase Project URL and 'anon public' key:

1.  **Go to your Supabase project dashboard.**
2.  **Navigate to 'Project Settings'** (usually a gear icon) and then **'API Keys'**.
3.  **Locate your 'Project URL'** and **'anon public' key**.
4.  **Copy these values.** You will paste them into the code cell below.

**Reasoning**:
Now that the user knows how to find their Supabase credentials, I will provide a code block for them to define the `SUPABASE_URL` and `SUPABASE_KEY` variables with placeholders that they need to replace.



In [ ]:
SUPABASE_URL = "YOUR_SUPABASE_PROJECT_URL"
SUPABASE_KEY = "YOUR_SUPABASE_ANON_PUBLIC_KEY"

print("Supabase URL and Key variables defined. Please replace the placeholders with your actual credentials.")

In [ ]:
from supabase import create_client, Client
from google.colab import files

# Initialize Supabase client
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

# --- Supabase Storage Configuration ---
# IMPORTANT: Replace 'your-bucket-name' with the actual name of your Supabase Storage bucket.
# You can create a new bucket in your Supabase project under Storage -> Buckets.
SUPABASE_BUCKET_NAME = "doc-upload"

# --- File Upload Functionality ---
print("Please upload the files you wish to store in Supabase. You can select multiple files.")

# This will open a file chooser dialog
uploaded_files = files.upload()

if not uploaded_files:
    print("No files were uploaded. Please try again.")
else:
    print(f"\nAttempting to upload {len(uploaded_files)} file(s) to Supabase bucket: '{SUPABASE_BUCKET_NAME}'")
    for filename, content in uploaded_files.items():
        try:
            # Upload the file content to Supabase Storage
            # The path will be the filename itself within the bucket
            response = supabase.storage.from_(SUPABASE_BUCKET_NAME).upload(filename, content)
            if response:
                print(f"Successfully uploaded '{filename}' to Supabase Storage.")
            else:
                print(f"Failed to upload '{filename}'. Supabase response was empty.")

        except Exception as e:
            print(f"Error uploading '{filename}': {e}")
            print("\n--- Troubleshooting Supabase Uploads ---")
            print("1. Ensure the `SUPABASE_BUCKET_NAME` variable above is set to an existing bucket name in your Supabase project.")
            print("2. Check your Supabase Storage RLS (Row Level Security) policies for the bucket. Ensure that the 'anon' role has `insert` and `select` permissions for uploads.")
            print("   Example RLS policy for public uploads (adjust as needed for security):")
            print("   CREATE POLICY \"Allow public uploads\" ON storage.objects FOR INSERT WITH CHECK (bucket_id = 'your-bucket-name');")
            print("   CREATE POLICY \"Allow public access\" ON storage.objects FOR SELECT USING (bucket_id = 'your-bucket-name');")
            print("3. Verify that your `SUPABASE_URL` and `SUPABASE_KEY` are correct and have not expired.")

Please upload the files you wish to store in Supabase. You can select multiple files.


Saving CE syllabus.pdf to CE syllabus.pdf

Attempting to upload 1 file(s) to Supabase bucket: 'doc-upload'
Storage endpoint URL should have a trailing slash.
Error uploading 'CE syllabus.pdf': {'statusCode': 403, 'error': Unauthorized, 'message': new row violates row-level security policy}

--- Troubleshooting Supabase Uploads ---
1. Ensure the `SUPABASE_BUCKET_NAME` variable above is set to an existing bucket name in your Supabase project.
2. Check your Supabase Storage RLS (Row Level Security) policies for the bucket. Ensure that the 'anon' role has `insert` and `select` permissions for uploads.
   Example RLS policy for public uploads (adjust as needed for security):
   CREATE POLICY "Allow public uploads" ON storage.objects FOR INSERT WITH CHECK (bucket_id = 'your-bucket-name');
   CREATE POLICY "Allow public access" ON storage.objects FOR SELECT USING (bucket_id = 'your-bucket-name');
3. Verify that your `SUPABASE_URL` and `SUPABASE_KEY` are correct and have not expired.


In [ ]:
import os
from getpass import getpass
!pip install -q langchain-google-genai
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# --- 0. SETUP API KEY ---
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google API Key: ")

# Initialize Model and Embeddings
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")
embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

# --- 1. PREPARE DATA FROM UPLOADED FILE ---
if 'content' not in locals() or not content:
    print("❌ No content found! Please run the file extraction code above first.")
else:
    print("--- Step 1: Splitting and Embedding Uploaded File ---")

    # Split the massive 'content' string into smaller chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,  # Characters per chunk
        chunk_overlap=200 # Overlap to maintain context between chunks
    )

    # Convert string chunks into Document objects
    # This replaces the manual 'docs = [Document(...)]' list in your example
    split_docs = text_splitter.create_documents([content])

    print(f"Split document into {len(split_docs)} chunks.")

    # Create Vector Store (FAISS) locally
    # This does the 'embedding' and 'add_documents' automatically
    vector_store = FAISS.from_documents(split_docs, embeddings)

    print("✅ Embeddings generated and stored in FAISS.")

    # --- 2. EXPLICIT RETRIEVAL ---
    # We ask a question relevant to YOUR PDF
    user_question = input("\nType your question about the document: ")
    # Default fallback if input is empty
    if not user_question:
        user_question = "Summarize the document."

    print(f"\n--- Step 2: Retrieving context for: '{user_question}' ---")

    # Get top 3 most similar chunks
    similar_docs = vector_store.similarity_search_with_score(user_question, k=3)

    retrieved_context = []
    print("Found the following relevant chunks:")
    for doc, score in similar_docs:
        # FAISS score: Lower is better (L2 distance), but usually we just want the content
        print(f"\n[Content Preview]: {doc.page_content[:100]}...")
        retrieved_context.append(doc.page_content)

    context_block = "\n\n".join(retrieved_context)

    # --- 3. GENERATION ---
    print(f"\n--- Step 3: Prompting Gemini ---")

    template = """You are a helpful assistant. Use only the following context to answer the question.

    Context from uploaded file:
    {context}

    Question:
    {question}
    """

    prompt = ChatPromptTemplate.from_template(template)
    chain = prompt | llm | StrOutputParser()

    final_answer = chain.invoke({
        "context": context_block,
        "question": user_question
    })

    print(f"\nGemini's Answer:\n{final_answer}")

ModuleNotFoundError: No module named 'langchain_community'

In [ ]:
!pip install -q langchain langchain-google-genai faiss-cpu langchain-community PyMuPDF python-docx pytesseract
!sudo apt install tesseract-ocr

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.3/53.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.6/426.6 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 477.4/477.4 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.3/233.3 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-

In [ ]:
import os
import fitz  # PyMuPDF
import docx
import pytesseract
from PIL import Image
from google.colab import files
import io
import numpy as np
from getpass import getpass

# LangChain & Vector Store Imports
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# --- 1. SETUP API KEY ---
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google API Key: ")

# --- 2. TEXT EXTRACTION FUNCTIONS ---
def extract_from_pdf(file_path):
    """Extracts text from a standard PDF."""
    text = ""
    with fitz.open(file_path) as doc:
        for page in doc:
            text += page.get_text()
    return text

def extract_from_docx(file_path):
    """Extracts text from a .docx file."""
    doc = docx.Document(file_path)
    return "\n".join([para.text for para in doc.paragraphs])

def extract_from_image(file_path):
    """Extracts text from an image using OCR."""
    return pytesseract.image_to_string(Image.open(file_path))

# --- 3. MAIN EXECUTION ---

# A. Trigger File Upload
print("\n--- Upload your document (PDF, DOCX, or Image) ---")
uploaded = files.upload()
raw_content = ""

# B. Process Uploaded File
for filename in uploaded.keys():
    print(f"\nProcessing: {filename}...")

    if filename.endswith('.pdf'):
        raw_content = extract_from_pdf(filename)
        # Check if PDF was scanned (empty text)
        if not raw_content.strip():
            print("⚠️ PDF appears to be an image/scanned. OCR might be needed (skipped for speed).")

    elif filename.endswith('.docx'):
        raw_content = extract_from_docx(filename)

    elif filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        raw_content = extract_from_image(filename)

    else:
        print("❌ Unsupported file format.")

# C. Embed & Vectorize (If content exists)
if raw_content:
    print("\n--- Generating Vector Embeddings ---")

    # 1. Split Text into Chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )
    docs = text_splitter.create_documents([raw_content])
    print(f"Created {len(docs)} text chunks.")

    # 2. Initialize Embeddings & Vector Store
    # using 'models/text-embedding-004' for better performance
    embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

    # Create FAISS Index from documents
    vector_store = FAISS.from_documents(docs, embeddings)
    print("✅ Vector Store Ready!")

    # D. Interactive Chat Loop
    llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")

    while True:
        query = input("\nAsk a question about your document (or type 'exit'): ")
        if query.lower() == 'exit':
            break

        # 3. Retrieve Context
        similar_docs = vector_store.similarity_search(query, k=3)
        context_text = "\n\n".join([d.page_content for d in similar_docs])

        # 4. Generate Answer
        template = """You are an assistant analyzing a user's document.
        Answer the question based ONLY on the following context.

        Context:
        {context}

        Question:
        {question}
        """

        prompt = ChatPromptTemplate.from_template(template)
        chain = prompt | llm | StrOutputParser()

        response = chain.invoke({"context": context_text, "question": query})

        print(f"\nGemini: {response}")
        print("-" * 50)

else:
    print("No text could be extracted from the file.")

ModuleNotFoundError: No module named 'langchain_google_genai'